In [2]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from global_model_periodic_energy_20 import LearnedSimulator_periodic
import torch.nn as nn
import torch.optim as optim
from scipy.spatial import Voronoi

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_float32_matmul_precision('high')
normalization_stats = {
    "velocity": {"mean": torch.tensor([0.0, 0.0]).to(device), "std": torch.tensor([1e-3, 1e-3]).to(device)},
    "acceleration": {"mean": torch.tensor([0.0, 0.0]).to(device), "std": torch.tensor([1e0, 1e0]).to(device)}
}

In [ ]:
checkpoint = torch.load("/home/jeanlienhard/Documents/Cell_GNN_clear/Cell_GNN/GNN for energy/GNN_20cells_8knn/Energy_with_perimeter_20cell_knn8_512_exp1/model_195.pth")
new_state_dict = {k.replace("_orig_mod.", ""): v for k, v in checkpoint.items()}
model = LearnedSimulator_periodic(num_dimensions=2, normalization_stats=normalization_stats, device=device,n_cells = 80)
model.load_state_dict(new_state_dict)
model.to(device)

LearnedSimulator_periodic(
  (graph_network): EnergyGNN(
    (edge_to_node): edgeToNode()
    (gnn_layer1): NodeGNN()
    (gnn_layer2): NodeGNN()
    (gnn_layer3): NodeGNN()
    (gnn_layer4): NodeGNN()
    (gnn_layer5): NodeGNN()
    (regressor): Sequential(
      (0): Linear(in_features=512, out_features=512, bias=True)
      (1): ReLU()
      (2): Linear(in_features=512, out_features=1, bias=True)
    )
  )
)

In [5]:
n_cells = 80
n_steps = 200    # nombre d'étapes simulées
dt = 0.1 # pas de temps
max_iter_per_step = 100  # iterations internes de LBFGS par step
lr = 0.5

In [ ]:
df = pd.read_csv("/home/jeanlienhard/Documents/Cell_GNN_clear/Cell_GNN/Data/random_data/80cells_4*4.csv")
x0 = df[df["step"] == 0].iloc[:n_cells][['x', 'y']].values.astype(np.float32)

# positions initiales comme variable optimisable
positions = torch.tensor(x0, dtype=torch.float32, device=device, requires_grad=True)
prev_prev_positions = positions.clone().detach()
prev_positions = positions.clone().detach()
# pour stocker la trajectoire
trajectory = [positions.detach().cpu().numpy()]

In [7]:
def make_periodic_copies(x_centers):
    x_offset, y_offset = 4.0, 4.0
    copies = [x_centers]
    for gx in range(-2,3):
        for gy in range(-2,3):
            if gx != 0 or gy != 0:
                shift = torch.tensor([gx*x_offset, gy*y_offset], device=x_centers.device)
                copies.append(x_centers + shift)
    x_full = torch.cat(copies, dim=0)
    return x_full.unsqueeze(1)


In [8]:
optimizer = torch.optim.LBFGS([positions], lr=lr, max_iter=max_iter_per_step, history_size=10, line_search_fn="strong_wolfe")
# scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
for step in range(n_steps):
    def closure():
        optimizer.zero_grad()
        x_centers = positions
       

        x_full = make_periodic_copies(x_centers)
        E_cell = model(x_full, n_cells)
        E_potential = E_cell.mean()
        
        acceleration = (positions.detach() - 2*prev_positions+prev_prev_positions) / (dt)
        accel_penalty = (acceleration**2).mean()
        # print(accel_penalty)
        #print(accel_penalty/E_potential)
        total_loss = 1e-2*E_potential + 1e2*accel_penalty

        total_loss.backward()
        for p in optimizer.param_groups[0]['params']:
            if p.grad is not None and not p.grad.is_contiguous():
                p.grad = p.grad.contiguous()
        return total_loss
    optimizer.step(closure)
    with torch.no_grad():
        acceleration = (positions.detach() - 2*prev_positions+prev_prev_positions) / (dt)
        accel_penalty = (acceleration**2).mean()
        print(torch.sqrt(accel_penalty))
    trajectory.append(positions.detach().cpu().numpy())
    prev_prev_positions = prev_positions.clone().detach()
    prev_positions = positions.clone().detach()



tensor(0.0014, device='cuda:0')
tensor(0.0014, device='cuda:0')
tensor(0.0018, device='cuda:0')
tensor(0.0022, device='cuda:0')
tensor(0.0016, device='cuda:0')
tensor(0.0025, device='cuda:0')
tensor(0.0052, device='cuda:0')
tensor(0., device='cuda:0')
tensor(6.9460e-09, device='cuda:0')
tensor(6.9460e-09, device='cuda:0')
tensor(6.9460e-09, device='cuda:0')
tensor(6.7481e-09, device='cuda:0')
tensor(7.3628e-10, device='cuda:0')
tensor(0., device='cuda:0')
tensor(6.9460e-09, device='cuda:0')
tensor(6.9460e-09, device='cuda:0')
tensor(0., device='cuda:0')
tensor(6.9460e-09, device='cuda:0')
tensor(6.9460e-09, device='cuda:0')
tensor(6.9460e-09, device='cuda:0')
tensor(6.7481e-09, device='cuda:0')
tensor(7.3628e-10, device='cuda:0')
tensor(6.9460e-09, device='cuda:0')
tensor(6.9460e-09, device='cuda:0')
tensor(6.9460e-09, device='cuda:0')
tensor(6.9460e-09, device='cuda:0')
tensor(6.9460e-09, device='cuda:0')
tensor(6.9460e-09, device='cuda:0')
tensor(6.9460e-09, device='cuda:0')
tensor(0

In [9]:
total_pos = []
step = 0
print(len(trajectory))
for traj in trajectory:
    x = torch.tensor(traj, requires_grad=True, dtype=torch.float32, device=device)
    x_centers = x.view(80, 2)
    full = make_periodic_copies(x_centers)
    for site_index in range(full.shape[0]):
            total_pos.append([step, site_index, full[site_index][0][0].item(), full[site_index][0][1].item()])
    step += 1
trajectories_df = pd.DataFrame(total_pos, columns=['step', 'site_index', 'x', 'y'])
trajectories_df.to_csv("/home/jeanlienhard/Documents/Cell_GNN_clear/Cell_GNN/GNN for energy/GNN_20cells_8knn/trajectories_80.csv", index=False)

201


In [10]:
x0_tensor = torch.tensor(x0.reshape(80,2), device=device, dtype=torch.float32)
x0_full = make_periodic_copies(x0_tensor).squeeze(1).detach().cpu().numpy()
df_initial = pd.DataFrame(x0_full, columns=['x','y'])
df_initial.to_csv("/home/jeanlienhard/Documents/Cell_GNN_clear/Cell_GNN/GNN for energy/GNN_20cells_8knn/positions_initiales_periodicite_80.csv", index=False)

In [11]:
final_x_full = make_periodic_copies(positions)
final_x_full_np = final_x_full.squeeze(1).detach().cpu().numpy() 
E_cell = model(final_x_full, n_cells)
print(E_cell,sum(E_cell),max(E_cell))
df = pd.DataFrame(final_x_full_np, columns=['x', 'y'])
df.to_csv("/home/jeanlienhard/Documents/Cell_GNN_clear/Cell_GNN/GNN for energy/GNN_20cells_8knn/positions_optimisees_periodicite_80.csv", index=False)

tensor([20.6011,  1.3140,  2.2285,  3.3700, 14.6311,  1.7859, 20.2366,  4.1248,
         2.2358, 12.1719,  1.8727,  0.6862,  3.4680,  8.2644, 19.4413,  8.9831,
         7.2176, 16.1105, 19.5120,  8.9143, 10.1541, 26.1319,  3.8067, 16.1235,
        10.4428,  3.3985, 13.6311,  1.1239,  0.9257,  1.2114,  3.1690, 13.3177,
         1.7331,  0.7718,  9.2697, 32.2415,  1.7853,  1.1318,  8.8150,  8.3081,
         3.6640,  6.2399,  4.0537,  5.7453, 13.5396,  2.4486,  1.7033,  6.3333,
         3.1336,  5.9964,  1.2299, 40.9370,  7.3265,  2.4165, 66.0721,  7.8342,
        26.0162,  9.3098,  4.0887,  0.9624, 33.4958,  4.0063, 12.7012,  2.2078,
        11.6464,  9.3277,  0.5386,  2.9647,  4.8514,  2.0637,  4.7948, 10.0165,
         0.8020,  2.4000,  3.3287,  1.8859, 12.0881, 12.7314, 68.8757, 11.1151],
       device='cuda:0', grad_fn=<SqueezeBackward1>) tensor(775.5551, device='cuda:0', grad_fn=<AddBackward0>) tensor(68.8757, device='cuda:0', grad_fn=<UnbindBackward0>)


In [12]:
def compute_polygon_area_and_perimeter(polygon):
    polygon = np.array(polygon)
    x = polygon[:, 0]
    y = polygon[:, 1]
    area = 0.5 * np.abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))
    perimeter = np.sum(np.linalg.norm(np.roll(polygon, -1, axis=0) - polygon, axis=1))
    return area, perimeter

def vornoi_area_and_perimeter(vor,target_indices):
    areas = []
    perimeters = []

    for idx in target_indices:
        region_index = vor.point_region[idx]
        region = vor.regions[region_index]
        if -1 in region or len(region) == 0:
            areas.append(1e-10)
            perimeters.append(1e-10)
            continue
        polygon = [vor.vertices[i] for i in region]
        area, perimeter = compute_polygon_area_and_perimeter(polygon)
        areas.append(area)
        perimeters.append(perimeter)          
    return areas,perimeters

In [13]:
def voronoi_loss(output,target_indices,accelerations,target_areas= 0.2,masse = 0.1, dt = 0.05,perimeter_target = 1.58):
    areas = []
    perimeters = []
    vor = Voronoi(output.cpu().detach().numpy())
    area,perimeter = vornoi_area_and_perimeter(vor,target_indices)
    areas.append(area)
    perimeters.append(perimeter)
    areas_tensor = torch.tensor(np.array(areas), dtype=torch.float32).to(device)
    perimeters_tensor = torch.tensor(np.array(perimeters),dtype=torch.float32).to(device)
    # areas_tensor = torch.stack(areas)
    # perimeters_tensor = torch.stack(perimeters)
    physics_loss = 0.02*(target_areas - areas_tensor)**2 + 0.005*(perimeters_tensor-perimeter_target)**2#+1e-5*(target_areas/(areas_tensor))**2
    # kinetic_loss = torch.sum((0.5*masse*dt**2)*(accelerations/(dt**2))**2,dim=-1)
    # print(physics_loss,kinetic_loss)
    return (physics_loss).squeeze(0)#+kinetic_loss

In [14]:
target_indices = np.arange(80)
res = 1e4*voronoi_loss(final_x_full.view(80*25,2),target_indices,None)
print(res, torch.max(res))

tensor([ 28.9985,   4.1174,   2.4627,   4.7054,  18.8596,   6.5517,   4.3600,
          2.5427,   2.3020,  14.5654,   0.6732,   0.8178,   3.8684,   8.6566,
         13.0238,   8.1934,   4.2373,  20.7324,  48.0782,   7.9877,   9.8538,
         40.8053,  19.4940,  22.9948,  11.7317,   1.0332,  13.4841,   5.4522,
          5.5688,   1.3769,   2.8562,  13.1437,   1.5949,   1.2143,   6.3791,
         64.4620,   1.1833,   4.0265,  13.4858,   9.7767,   3.0689,   6.1741,
          0.5364,   4.0461,  19.1421,   0.9717,   1.4883,   0.8565,  15.8758,
          5.4243,   0.6220,  30.4542,  11.1573,   0.5775, 106.6089,   7.8105,
         19.6497,   1.8103,   6.8467,   1.1386,  46.4235,   1.3592,  16.5696,
          3.5556,   5.4962,  17.9215,   4.4010,   0.3112,   2.2829,   1.7037,
         10.6582,   9.1630,   2.7328,   5.1146,   0.4093,   5.3478,  21.7712,
          9.8257,  70.1782,  23.9436], device='cuda:0') tensor(106.6089, device='cuda:0')
